# Demonstrating `uncertainty` module

In this notebook we show the how to use and apply the classess and method in the sub modules of the `uncertainty` module this include:
- Importing uncertainty information
- Filtering uncertain parameters
- selecting and changing probability distribution information of uncertain parameters

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import pandas as pd
import numpy as np
sys.path.append('..')
from pulpo import pulpo
from pulpo.utils.uncertainty import preparer, processor, plots, gsa
from pulpo.utils import optimizer, saver

## 0. Setup Background and Foreground Databases

### 0.1: Background Database (Ecoinvent 3.10 cutoff)

In [ ]:
from pathlib import Path
import bw2data as bd
import bw2io as bi

PROJECT = "ammonia_final"
DB_NAME = "ecoinvent-3.10-cutoff"
CRED_PATH = Path("/Users/hausslingbhl/Library/CloudStorage/OneDrive-UniversiteitLeiden/01_Administration/02_VITO/04_Systems/credentials.txt")

def read_credentials(path: Path):
    if not path.is_file():
        raise FileNotFoundError(f"Couldn't find credentials file at: {path.resolve()}")
    creds = {}
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        # allow "key=value" or "key: value" or "key value"
        for sep in ("=", ":", " "):
            if sep in line:
                k, v = line.split(sep, 1)
                creds[k.strip().lower()] = v.strip()
                break
    if "username" not in creds or "password" not in creds:
        raise ValueError("credentials.txt must contain 'username' and 'password'.")
    return creds["username"], creds["password"]

# 1) Ensure project exists / is selected
bd.projects.set_current(PROJECT)

# 2) Import ecoinvent 3.10 cutoff if missing
if DB_NAME in bd.databases:
    print(f"Database '{DB_NAME}' already exists in project '{bd.projects.current}'.")
else:
    username, password = read_credentials(CRED_PATH)
    bi.import_ecoinvent_release(
        version="3.10",
        system_model="cutoff",  # "cutoff", "apos", "consequential", or "EN15804"
        username=username,
        password=password,
    )
    print(f"Database '{DB_NAME}' installed successfully.")

### 0.2: Foreground Database (Ammonia Production)

In [ ]:
# Path to your Excel file
excel_path = r"data/ammonia.xlsx"
fg_db_name = "ammonia"

if fg_db_name in bd.databases:
    print(f"Database '{fg_db_name}' already exists in project '{bd.projects.current}'.")
else:
    # 1. Import the Excel file
    fg_db = bi.ExcelImporter(excel_path)

    # 2. Apply strategies to clean and prepare the data
    fg_db.apply_strategies()

    # 3. Match the foreground database to itself (for internal links)
    fg_db.match_database(fields=["name", "unit", "reference product", "location"])

    # 4. Match to ecoinvent technosphere (use your actual ecoinvent db name)
    fg_db.match_database(
        "ecoinvent-3.10-cutoff",
        fields=["name", "unit", "location", "reference product"]
    )

    # 5. Match to ecoinvent biosphere (biosphere db is usually auto-named, check with list(bd.databases))
    biosphere_db = [db for db in bd.databases if "biosphere" in db and "3.10" in db][0]
    fg_db.match_database(
        biosphere_db,
        fields=["name", "categories", "location"]
    )

    # 6. Check statistics (should have 0 unlinked exchanges)
    fg_db.statistics()

    # 7. Write the database to disk
    fg_db.write_database()
    print(f"Database '{fg_db_name}' installed successfully.")

### 0.3: Install premise for GWP characterization factors

**Note:** Before executing the next cell, make sure to install `premise` via:
```bash
pip install premise
```

This package is required for adding updated GWP characterization factors to the project.

In [ ]:

# Check if the IPCC 2021 method already exists before adding premise GWP
import bw2data as bd

target_method = ('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')

if target_method in bd.methods:
    print(f"Method '{target_method}' already exists in the project.")
else:
    print(f"Method '{target_method}' not found. Adding premise GWP characterization factors...")
    from premise_gwp import add_premise_gwp
    add_premise_gwp()
    print("Premise GWP characterization factors added successfully.")

### 0.4: Install IPCC 2013 GWP method with uncertainty

This method has been obtained and adapted from: https://github.com/aleksandra-kim/gwp_uncertainties

In [ ]:
from bw2io.package import BW2Package

target_method = ('IPCC 2013', 'climate change', 'global warming potential (GWP100)', 'uncertain')

if target_method in bd.methods:
    print(f"Method '{target_method}' already exists in the project.")
else:
    print(f"Method '{target_method}' not found. Importing IPCC 2013 GWP method with uncertainty...")
    BW2Package.import_file("data/ipcc_uncertain.bw2package")
    print("IPCC 2013 GWP with uncertainty characterization factors added successfully.")


## 1. Case Study Setup: Ammonia Production System

In [ ]:
def setup_ammonia_case_study():
    """
    Set up the ammonia production case study with PULPO configuration.
    
    Returns:
        tuple: (project, database, method, directory) configuration parameters
    """
    project = "ammonia_final"
    database = ["ecoinvent-3.10-cutoff", "ammonia"]
    method = "('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')"
    directory = "develop_tests"
    
    return project, database, method, directory

def create_pulpo_worker(project, database, method, directory):
    """
    Create and initialize a PULPO optimizer instance.
    
    Args:
        project (str): PULPO project name
        database (str): Database name
        method (str): LCIA method specification
        directory (str): Working directory path
        
    Returns:
        pulpo.PulpoOptimizer: Configured PULPO worker instance
    """
    # Create PulpoOptimizer instance
    pulpo_worker = pulpo.PulpoOptimizer(project, database, method, directory)
    pulpo_worker.intervention_matrix = "ecoinvent-3.10-biosphere"
    
    # Import LCI data
    pulpo_worker.get_lci_data()
    
    return pulpo_worker

def get_single_process(worker, query, prefer_locations=("RER", "Europe", "GLO")):
    """
    Deterministic process retrieval to avoid order issues.
    
    Args:
        worker: PULPO optimizer instance
        query (str): Process name query
        prefer_locations (tuple): Preferred locations in order of preference
        
    Returns:
        Process object from the database
    """
    matches = worker.retrieve_processes(processes=query)
    if not matches:
        raise ValueError(f"No process found for query: {query}")
    for loc in prefer_locations:
        for p in matches:
            if getattr(p, "location", None) == loc or loc in str(p):
                return p
    return sorted(matches, key=lambda x: str(x))[0]

def define_ammonia_problem(pulpo_worker):
    """
    Define the ammonia production optimization problem with streamlined configuration.
    
    Args:
        pulpo_worker: PULPO optimizer instance
        
    Returns:
        tuple: (choices, demand) for the optimization problem
    """
    # Choice definitions with capacities bound per-label
    choice_config = {
        "biogas": {
            "processes": [
                "anaerobic digestion of agricultural residues",
                "anaerobic digestion of sequential crop",
            ],
            # 2030 EU-27 potentials from biomethane shares (38 bcm total; 24% ag, 21% sequential),
            # converted to raw biogas assuming ~57% CH₄ → 16.0 & 14.0 bcm ≈ 1.60e10 & 1.40e10 m³/yr.
            "capacities": [1.60e10, 1.40e10],
        },
        "biomethane": {
            "processes": [
                "upgrading water scrubbing (CCS)",
                "upgrading water scrubbing",
                "upgrading chemical scrubbing",
                "upgrading chemical scrubbing (CCS)",
            ],
            "capacities": [1e20, 1e20, 1e20, 1e20],
        },
        "methane": {
            "processes": ["market for methane fg", "market for biomethane"],
            "capacities": [1e20, 1e20],
        },
        "heat": {
            "processes": ["heat from methane", "heat from methane (CCS)", "heat from hydrogen"],
            "capacities": [1e20, 1e20, 1e20],
        },
        "hydrogen": {
            "processes": [
                "methane pyrolysis",
                "steam methane reforming",
                "steam methane reforming (CCS)",
                "plastics gasification",
                "plastics gasification (CCS)",
                "alkaline electrolysis",
                "PEM electrolysis",
            ],
            # Methane pyrolysis capped to 10,000 t H2/yr (= 1.0e7 kg/yr); others left high for now.
            # "capacities": [3.0e8, 1e20, 1e20, 1e20, 1e20, 1e20, 1e20],
            "capacities": [1e20, 1e20, 1e20, 1e20, 1e20, 1e20, 1e20],
        },
        "electricity": {
            "processes": [
                "grid electricity",
                # "wind onshore electricity",
            ],
            "capacities": [1e20],#, 5e10], # Placeholder cap for wind onshore
        },
        "ammonia": {
            "processes": [
                "steam reforming, integrated",
                "steam reforming, integrated (CCS)",
                "nitrogen + hydrogen",
            ],
            "capacities": [1e20, 1e20, 1e20],
        },
    }

    # Build choices with deterministic mapping
    choices = {}
    for category, cfg in choice_config.items():
        labels, caps = cfg["processes"], cfg["capacities"]
        if len(labels) != len(caps):
            raise ValueError(f"Length mismatch in '{category}': {len(labels)} labels vs {len(caps)} capacities")
        choices[category] = {get_single_process(pulpo_worker, lbl): cap for lbl, cap in zip(labels, caps)}

    # Demand (EU ammonia, kg/yr)
    demand_process = get_single_process(pulpo_worker, "market for ammonia")
    demand = {demand_process: 17.1e9}  # ~17.1 Mt/yr (EU)

    # Additional upper bounds (shared resources / feedstocks)
    waste_pp = get_single_process(pulpo_worker, "treatment of waste PP")
    waste_ps = get_single_process(pulpo_worker, "treatment of waste PS")
    ccs_process = get_single_process(pulpo_worker, "CCS 200km pipeline 1000m deep")

    upper_bounds = {
        waste_pp: 1e20,#1.875e9,  # 25% of ~7.5 Mt PP post-consumer waste ≈ 1.875 Mt/yr
        waste_ps: 1e20,#3.25e8,   # 25% of ~1.3 Mt PS waste ≈ 0.325 Mt/yr
        ccs_process: 1e20,#5.0e9, # 5 MtCO2/yr (10% of EU-27 2030 NZIA target)
    }
    
    # Instantiate the optimization problem
    pulpo_worker.instantiate(demand=demand, choices=choices, upper_limit=upper_bounds)
    
    return choices, demand

def solve_and_summarize(pulpo_worker, file_name='ammonia_results'):
    """
    Solve the optimization problem and summarize results.
    
    Args:
        pulpo_worker: PULPO optimizer instance
        file_name (str): Filename for results (optional)
        
    Returns:
        dict: Extracted results data
    """
    # Solve optimization problem
    pulpo_worker.solve()
    
    # Extract and summarize results
    result_data = pulpo_worker.extract_results()
    pulpo_worker.summarize_results(zeroes=True)
    
    return result_data

In [ ]:
# Set up the ammonia case study
project, database, method, directory = setup_ammonia_case_study()

# Create and initialize PULPO worker
pulpo_worker = create_pulpo_worker(project, database, method, directory)

# Define the optimization problem
choices, demand = define_ammonia_problem(pulpo_worker)

# Solve the problem and get results
# result_data = solve_and_summarize(pulpo_worker, file_name='ammonia_test')

print(f"✅ Setup complete: {sum(len(c) for c in choices.values())} alternatives across {len(choices)} categories")

## 2. Filtering out negletable uncertain parameters

We only consider uncertainty in the $B$ and $Q$ parameter matrizes. The scaling vector is given by the optimal solution.

The parameter filter, aims to reduce the set of uncertain parameters to reduce the computational effort for the GSA or for CC optimization. 

There is however an inherent risk when reducing the set of parameters before using chance constraint optimization, since relevant uncertain parameters at the non deterministic solution might be filtered out, which can be significant at higher probability levels.

In [ ]:
paramfilter = preparer.ParameterFilter(
    lci_data=pulpo_worker.lci_data, 
    choices = choices,
    demand = demand,
    method = method
    )

Using the basic scaling vector which only includes the optimal choices in the scaling vector

In [ ]:
scaling_vector_series = paramfilter.prepare_scaling_vector(scaling_vector_strategy='naive', result_data=result_data)

Using the scaling vector constructed from all choices

In [ ]:
scaling_vector_series = paramfilter.prepare_scaling_vector(scaling_vector_strategy='constructed_demand')

Compute the LCA scores and return the characterized inventory to be used for the filtering

In [ ]:
lca_score, characterized_inventory = paramfilter.compute_LCI_LCIA(scaling_vector_series)

Plot the largest contributors

In [ ]:
plots.plot_top_characterized_processes(pulpo_worker.lci_data['process_map_metadata'], characterized_inventory, method, top_amount=19)

Filtering out the inventoryflows $B_{i,j}$ that have a neglectable impact

In [ ]:
cutoff = 0.000019 # ATTN: Change to 0. to include all parameters, this is needed to include the CCS biosphere flows else they are filtered out
# cutoff = 0.
filtered_inventory_indcs = paramfilter.filter_inventoryflows(characterized_inventory, lca_score, cutoff)

In [ ]:
filtered_characterization_indcs = paramfilter.filter_characterization_factors(filtered_inventory_indcs)

### 2.1. Filtering method
Alternativly all the methods can be called with the `apply_filter` method.

In [ ]:
filtered_inventory_indcs, filtered_characterization_indcs = paramfilter.apply_filter(
    scaling_vector_strategy='constructed_demand',
    cutoff=0.,#0.001,
    plot_results=True,
    plot_n_top_processes=19
)

## 3. Getting the uncertainty of the parameter values

### 3.1. Importing the uncertainty data

Imports uncertainty data for the intervention flows and the characterization factors from the databases and Brightway project. It also creates uncertainty data (with unspecified distributions) for the variables bounds specified for the pulpo instance.

Extracts the metadata containing the uncertainty information to the filtered intervention flows and seperate the metadata into the parameters with and without defined uncertainty information

In [ ]:
uncertainty_importer = preparer.UncertaintyImporter(
    lci_data=pulpo_worker.lci_data, 
    bw_databases=database, 
    LCIA_method=method,
)
uncertainty_data = uncertainty_importer.import_uncertainty_data(
    if_indcs=filtered_inventory_indcs,
    cf_indcs=filtered_characterization_indcs,
    choices=pulpo_worker.choices,
    upper_limit=pulpo_worker.upper_limit,
    lower_limit=pulpo_worker.lower_limit,
    upper_elem_limit=pulpo_worker.upper_elem_limit,
    upper_imp_limit=pulpo_worker.upper_imp_limit,
)

### 3.2. Apply strategies to define missing uncertainty data

#### 3.2.1. Intervention flows

In [ ]:
print(database)

Apply the triangular strategy using bound interpolation to the missing intervention uncertainty parameters in the background database

In [ ]:
if_bg_triangular_strategy = processor.TriangularBoundInterpolationStrategy(
    uncertain_param_type='If',
    uncertain_param_subgroup='ecoinvent-3.10-cutoff',
    noise_interval={'min':.1, 'max':.1}
    )
if_bg_triangular_strategy.assign(uncertainty_data)

Apply the Uniform strategy for the uncertainty parameters in the foreground database

In [ ]:
if_fg_uniform_strategy = processor.UniformBaseStrategy(
    uncertain_param_type='If',
    uncertain_param_subgroup='ammonia',
    upper_scaling_factor = .5,
    lower_scaling_factor = .5,
    noise_interval={'min':.2, 'max':.2}
)
if_fg_uniform_strategy.assign(uncertainty_data)

Set expert judgement uncertainties to a few selected intervention flows:

In [ ]:
# ATTN: It might be better to not call an index directly but search for the process and the intervention flow, just not sure how
# This can be done in a step before which returns the indices and can be called like defined here

CCS_expert_uncertainty_info = {
    (715, 23523): {'minimum':.01 ,'maximum':.5, 'uncertainty_type':4}, # CCS 200km pipeline 1000m deep | CO2 stored | RER --- Carbon dioxide, non-fossil | ('air',)
    (80, 23561): {'minimum':.0002 ,'maximum':.002, 'uncertainty_type':4}, # transport, supercritical carbon dioxide | transport, pipeline, supercritical carbon dioxide, 200 km without recompression | RER --- Carbon dioxide, fossil | ('air', 'urban air close to ground')
    }
if_fg_expert_knowledge_strategy = processor.ExpertKnowledgeStrategy(
    uncertain_param_type='If',
    uncertain_param_subgroup='ammonia',
    prob_metadata=CCS_expert_uncertainty_info
)
if_fg_expert_knowledge_strategy.assign(uncertainty_data)


Check if the uncertainty data has been assgined correctly to the flows

In [ ]:
processor.rename_metadata_index(
    pd.DataFrame.from_records(uncertainty_data['If']['ammonia']['defined']).T, 
    pulpo_worker.lci_data, 
    'intervention_flow'
    )


In [ ]:

processor.rename_metadata_index(
    pd.DataFrame.from_records(uncertainty_data['If']['ammonia']['defined'])[list(CCS_expert_uncertainty_info.keys())].T, 
    pulpo_worker.lci_data, 
    'intervention_flow'
    )


#### 3.2.2. Characterization factors

Apply the triangular strategy using predefined scaling factors to the missing uncertainty parameters

In [ ]:
cf_triangular_strategy = processor.TriangluarBaseStrategy(
    uncertain_param_type='Cf',
    uncertain_param_subgroup=method,
    upper_scaling_factor = 0.15,
    lower_scaling_factor = 0.15,
    noise_interval={'min':.1, 'max':.1}
)
cf_triangular_strategy.assign(uncertainty_data)

#### 3.2.3. Variable bounds

Defining the uncertainty information for the variable bounds

Apply triangular uncertainty strategy with upper and lower scaling factor to the variable bounds (any other strategy is also possible)

In [ ]:
var_bound_upper_strategy = processor.TriangluarBaseStrategy(
    uncertain_param_type='Var_bounds',
    uncertain_param_subgroup='upper_limit',
    upper_scaling_factor=.3,
    lower_scaling_factor=.3,
    noise_interval={'min':.2, 'max':.1}
)
var_bound_upper_strategy.assign(uncertainty_data)

#### 3.2.4. All at once

The strategies can all be applied at once if they are passed to the `apply_uncertainty_strategies` method.

In the future this can also easily be rewritten to pass a json file or dict containing the set up for each uncertainty strategy

In [ ]:
unc_strategies = [
    processor.TriangularBoundInterpolationStrategy(
        uncertain_param_type='If',
        uncertain_param_subgroup='ecoinvent-3.10-cutoff',
        noise_interval={'min':.1, 'max':.1}
    ),
    
    processor.UniformBaseStrategy(
        uncertain_param_type='If',
        uncertain_param_subgroup='ammonia',
        upper_scaling_factor = .5,
        lower_scaling_factor = .5,
        noise_interval={'min':.2, 'max':.2}
    ),
    processor.TriangluarBaseStrategy(
        uncertain_param_type='Cf',
        uncertain_param_subgroup=method,
        upper_scaling_factor = 0.15,
        lower_scaling_factor = 0.15,
        noise_interval={'min':.1, 'max':.1}
    ),
    processor.TriangluarBaseStrategy(
        uncertain_param_type='Var_bounds',
        uncertain_param_subgroup='upper_limit',
        upper_scaling_factor=.3,
        lower_scaling_factor=.3,
        noise_interval={'min':.2, 'max':.1}
    ),
    
    processor.ExpertKnowledgeStrategy(
        uncertain_param_type='If',
        uncertain_param_subgroup='ammonia',
        prob_metadata={
            (715, 23523): {'minimum':.01 ,'maximum':.5, 'uncertainty_type':4}, # CCS 200km pipeline 1000m deep | CO2 stored | RER --- Carbon dioxide, non-fossil | ('air',)
            (80, 23561): {'minimum':.0002 ,'maximum':.002, 'uncertainty_type':4}, # transport, supercritical carbon dioxide | transport, pipeline, supercritical carbon dioxide, 200 km without recompression | RER --- Carbon dioxide, fossil | ('air', 'urban air close to ground')
        }
    )
    ]

In [ ]:
processor.apply_uncertainty_strategies(
    uncertainty_data,
    unc_strategies
    )



## 4. Filtering, Importing and Applying strategies directly on Pulpo Worker

The uncertainty data can be filtered and imported and strategies applied directly with the pulpo worker, shown below. In the following subsections the method called in the pulpo worker wrapper methods are shown and explained

In [ ]:
pulpo_worker.import_and_filter_uncertainty_data(
    cutoff=0.,#0.000019,
    scaling_vector_strategy='constructed_demand',
    plot_results=True,
    plot_n_top_processes=19
)

The uncertainty strategies can be directly performed on the pulpo worker if the strategies are passed to the pulpo worker, else a standard trigional distribution is used

In [ ]:
pulpo_worker.apply_uncertainty_strategies(strategies=unc_strategies)

## 5. Define the global sensitivity problem

In [ ]:
from SALib.sample import sobol as sample_method
from SALib.analyze import sobol as SA_method

The global sensitivity analysis can be called directly from the pulpo worker

In [ ]:
# ATTN: Change method to string, as gsa has problems with dict ...
pulpo_worker.method = "('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')"

In [ ]:
'''
pulpo_worker.run_gsa(
    result_data=result_data,
    sample_method=sample_method,
    SA_method=SA_method,
    sample_size=100,
    plot_gsa_results=True
)
'''

In [ ]:
# ATTN: Change back method to dict ...
pulpo_worker.method = {"('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')": 1}

## 5. Chance constraint optimization

### 5.1. Formulate CC problem

There are different ways to formulate the chance constrain optimization problem:
- CC the objectives (uncertainties in the environmental costs)
- CC the variable bounds (also only individual bounds can be chance constrained):
    - scaling variable bounds (`UPPER_LIMT` and `LOWER_LIMIT`)
    - Inventory limit (`UPPER_INV_LIMIT` and `UPPER_IMP_LIMIT`)

The chance constraints can only be applied if the corresponding parameters have true uncertainty information specified.


Transform all uncertain parameter distributions to gausian distributions, since the CC formulation requires normal distributions

In [ ]:
normal_uncertainty_data = processor.transform_to_normal(
    uncertainty_data,
    sample_size=100, 
    plot_distribution=False
    )

Compute the standard deviation and mean for the environmental costs, assuming normally distributed intervention flows and characterization factors and using the L1 Norm to approximate the Square root.

$$
    \sqrt{\sum_j s_j^2 \cdot \sigma_{q_hb_j}^2}
$$

In [ ]:
normal_metadata_env_cost = optimizer.compute_L1_env_cost_mean_var(
    normal_uncertainty_data= normal_uncertainty_data,
    lci_data=pulpo_worker.lci_data,
    method=method,
    plot_analysis_support_plots=False
)

Extract the variable bounds for which we want to impose CC for

In [ ]:
CC_var_bounds = ['upper_limit'] # ATTN: Add other variable bounds here if needed
normal_metadata_var_bounds = {var_bound:normal_uncertainty_data['Var_bounds'][var_bound]['defined'] for var_bound in CC_var_bounds}


### 5.2. Solve one CC problem Pareto point

The CC problem is solved as a Pareto optimization using on environmental impact category and the probability level underlying the chance constraints.

Update the environmental costs and the variable bounds of the optimization problem based on the CC formulation. We are using a short cut here, where we compute the updated environmental costs and bound parameters based on the linear L1 gausian CC formulation

$$
    \big( \mu_{q_hb_j} + \Phi^{-1}(\lambda_{z_h}) \cdot \sigma_{q_hb_j} \big) 
$$

In [ ]:
optimizer.apply_CC_formulation(
    lambda_level=0.8,
    normal_metadata_env_cost=normal_metadata_env_cost,
    normal_metadata_var_bounds=normal_metadata_var_bounds,
    model_instance=pulpo_worker.instance,
) 

Solve single Pareto point

In [ ]:
pulpo_worker.solve()
pulpo_worker.summarize_results()
results = pulpo_worker.extract_results(extractparams=True)

### 5.3. Solve the CC Problem using Epsilon Constraint method

Solve for an array of lambda epsilon constraints

In [ ]:
lambda_epsilon_array = np.linspace(0.5,1, 3, endpoint=False)
results_CC = {}
for lambda_level in lambda_epsilon_array:
    print(f'Solving for lambda = {lambda_level}')
    optimizer.apply_CC_formulation(
        lambda_level=lambda_level,
        normal_metadata_env_cost=normal_metadata_env_cost,
        normal_metadata_var_bounds=normal_metadata_var_bounds,
        model_instance=pulpo_worker.instance,
    ) 
    pulpo_worker.solve()
    results_CC[lambda_level] = pulpo_worker.extract_results(extractparams=True)
    pulpo_worker.summarize_results(zeroes=True)

Plot Pareto front

In [ ]:
plots.plot_pareto_front(
    result_data_CC=results_CC, 
    cutoff_value=0.01, 
    method="\n".join(method.split("'")[1::2]), 
    process_map_metadata=pulpo_worker.lci_data['process_map_metadata'], 
    bbox_to_anchor=(0.65,-3.5),
    cmap_name='tab20'
    )


Compare the Pareto points

In [ ]:
saver.compare_subsequent_paretosolutions(results_CC, choices, method)

## 5.4. Directly using the Pulpo Worker 

This can all be done in one step by calling the `create_CC_formulation` in the pulpo worker

In [ ]:
normal_metadata_env_cost, normal_metadata_var_bounds = pulpo_worker.create_CC_formulation(CC_env_cost=True, CC_var_bounds=['upper_limit'])

Using the pulpo worker

In [ ]:
results = pulpo_worker.solve_CC_problem(0.8, normal_metadata_env_cost, normal_metadata_var_bounds)

Using the Pulpo worker

In [ ]:
lambda_epsilon_array = np.linspace(0.5,1, 3, endpoint=False)
results_CC = pulpo_worker.solve_CC_problem(lambda_epsilon_array, normal_metadata_env_cost, normal_metadata_var_bounds, plot_results=True)

# 6. Case study: Integrating epistemic uncertainty factors

## 6.1. Changes in the ammonia database
- Added 0.1 kg 'Carbon dioxide, non-fossil' in 'CCS 200km pipeline 1000m deep'
- Changed "electricity, high voltage	grid electricity	RER	kilowatt hour	('ammonia', '49891e5d8da5e6cf0264c313f6a66376_copy1')" to be the aggregated activity of the ecoinvent version with the 10 most contributing biosphere flows. Then multiplying with 0.25, which represents a 75% decarbonized electricity mix. These biosphere flows are then used to add triag distribution 


Recompute the PULPO worker

In [ ]:
# Create and initialize PULPO worker
pulpo_worker = create_pulpo_worker(project, database, method, directory)
# Define the optimization problem
choices, demand = define_ammonia_problem(pulpo_worker)

## 6.2. The epistemic uncertainty
### 6.2.1. Biomass (agriculture residues and sequential crops) to biugas:
- Intrinsic factors: (directly linked to the technology/product): 
    - efficiency and functionality change: 
        - functional change: Intercropping is meant to bind the nitrogen in the soil and degrade on the soil to capture nutrients and improve the soil structure, etracting it will create environmental losses not accounted for here.
        - functional change: Depending on what biogas/methane is used for, e.g., hydrogen source, heating source and other competing chemical service functions it might be used for that instead, which might force production to use natural gas again.
        - production efficiency: 
            - uptake efficiency linked to molar equivalency of how much biogas can be produced from a kilogram of biomass 
            - non-linear correlation in production between, e.g.,  loading of plant and reaction efficiency, co-products, seperation-efficiency (linked to gas composiiton and by-products) etc.
    - learning and degradation: Teaching farmers to valorize 
    - spatial heterogeneity: transportation distances from field to plant
    - infrastructure requirements: decentralized biogas plants required, if not available much further transports rqequired and losses
    - resource criticality: Biomass will be required in many different areas beyond biogas and biogas/methane will be required for many other applications beyond industrial heating, Hydrogen source
    - end-of-life processes: "linked to assumptions in CCS" biomass is both the input and the output from a system perspective as ammonia is used for fertilizers which produces the biomass, it becomes a cyclic system.
- Indirect factors (system interactions): 
    - displacement of incumbent technologies: 
        - Only relevant if it displaces natural gas
        - Changes in the background system during the transformation, linked to other transformations, shocks, political developments, temporal effects etc.
    - behavioural responses and rebound:
        - Increased produciton of ammonia due to novel production routes or carbon deficit trading enabling more harmfull production technologies to remain operative.
        - By extracting even more nutrients from the soil and the area by also completely using all residues etc. leads to more linear production systems quicker soil degredation which in turn requires more intensive production methods, e.g., more fertilizer or more land-use
        - Biogas/methane will be required in many places which might lead to a switch back to natural gas, locking in on HB as the only way to produce NH3
    - allocation factors: allocation of production activities (harvesting etc.) to residues/manure, as from a system perspective they are all within the food system, allocating a small share to residues makes residues attractive for production intensive processes such as ammonia production instead of keeping 
    - supply chain effects: Moving to more organic production system or vegan diets, required to remain within ecological limits, will reduce the availability of second generation biomass
- _External factors_ (contextual): 
    - exogenous system shocks: 
        - natural desaster influencing biomass production and requiring more biomass to be used for food compared to feed, fuel.
        - wars and conflicts intervening into global food supply chains leading to more localized production (less residues available, more land-use, etc.) 
    - policy/regulatory developments: Shifting towards less intensive farming or organic farming
    - macroeconomic dynamics: crisis driving food and biomass prices up

#### 6.2.1.1. Translating into probabilistic constraints:
- Unknown changes in background system (indirect factors and external factors): modelled by interpolating the background uncertainties and adding uncertainty to it?
- Intrinsic and indirect factors of biomass-biogas-ammonia production system modelled as high uncertainty on all emissions and resource use (Biosphere) with uniform distribution
- ==Could also add emissions and specific uncertainty distirbutions to the different biomass production routes representing the intrinsic and indirect factors==

### 6.2.2. Carbon capture and storage:
- _Intrinsic factors_ (directly linked to the technology/product): 
    - efficiency and functionality change: 
        - Leaking/Efficiency from seperation of CO2 from biogas over transport to final storage
        - Storage stability and leakage 
    - learning and degradation, 
    - spatial heterogeneity, 
    - infrastructure requirements: 
        - Requires CO2 pipelines from every decentralized anaerobic digestion plant to storage locations 
        - Requires storage places within 100 km of anaerobic digestions plants or close to ammonia plants (very high amounts)
    - resource criticality
    - end-of-life processes: 
- _Indirect factors_ (system interactions): 
    - displacement of incumbent technologies, 
    - behavioural responses and rebound: pure CO2 will become a valuance resource in the future there is a probability that the purified CO2 will not be stored in the end but sold or used as a feedstock for another process
    - allocation factors: Based on if CO2 will not be used somewhere else, the allocation of the burden to the CCS might be wrong
    - supply chain effects.
- _External factors_ (contextual): exogenous system shocks, policy/regulatory developments, macroeconomic dynamics.

#### 6.2.2.1. Translating into probabilistic constraints:
- Unknown changes in background system (indirect factors and external factors): modelled by interpolating the background uncertainties and adding uncertainty to it?
- Intrinsic and indirect factors of biomass-biogas-ammonia-CCS production system modelled as high uncertainty on all emissions and resource use (Biosphere) with uniform distribution
- Additional CO2 flow added to the storage and transport to represent the intrinsic and indirect factors of CCS
- Expert judgement on the uncertainty of CCS added on both the CO2 emissions during transport and storage


### 6.2.3. Electricity and electrochemical technologies
- _Intrinsic factors_ (directly linked to the technology/product): 
    - efficiency and functionality change, 
    - learning and degradation, 
    - spatial heterogeneity, 
    - infrastructure requirements: Shifting towards renewable sources requires massive infrastrucutre change (production, sotrage, disgribution, etc.), which we are far from and not aiming towards
    - resource criticality: Requires a lot of rare earths and metals which might not be available due to production limitation
    - end-of-life processes: Circularity of renewable production technologies
- _Indirect factors_ (system interactions): 
    - displacement of incumbent technologies, 
    - behavioural responses and rebound:
        - More electricity available will increase the use of electicity if it is not hard constraints, which will lead to the grid still containing fossil energy production
    - allocation factors, 
    - supply chain effects.
- _External factors_ (contextual): 
    - exogenous system shocks:  wars, anti-colonial movements in the global south might lead to increase resource criticality 
    - policy/regulatory developments: infrastrucutre building is heavily dependent on policy and the current strong shift to far-right and climate change deniers and fossil-lobby adjecent politicians shows a weakening of policies and targetting renewable electricity as something bad, e.g., German AfD
    - macroeconomic dynamics: Fossil based electricity is still more profitable, which leads to market based investments to continue funding fossil infrastructure

#### 6.2.3.1. Translating into probabilistic constraints:
- Unknown changes in background system (indirect factors and external factors): modelled by interpolating the background uncertainties and adding uncertainty to it?
- Intrinsic and indirect factors of electicity-ammonia production system modelled as high uncertainty on all emissions and resource use (Biosphere) with uniform distribution
- The indirect and intrinsic factors linked to the availability of renewable electricity in the grid modelled by modelling the average grid as aggregated process, linked to its decarbonization and then adding a triangular probability distribution with 90% decarbonized as mode "Wind" as min and max as 75% decarbonized

### 6.2.4. Ammonia production
- No end of life of ammonia is considered, though the majority of the impact of ammonia as with other chemicals are in the use and end-of-life of the chemical. As well as the societal relevance is deeply linked to tha application and therefore the allocation of limit or burdens.
- Technological lock-ins with centralized Haber Bosch processes, risks due to high dependency
- Rebound effects: If novel NH3 production processes are introduced it could just lead to a new product "green ammonia" being used in new sectors not substituting essential activites but creating new products or services or increasing their amount.


In [ ]:
unc_strategies = [
    processor.TriangularBoundInterpolationStrategy(
        uncertain_param_type='If',
        uncertain_param_subgroup='ecoinvent-3.10-cutoff',
        noise_interval={'min':.1, 'max':.1}
    ),
    
    processor.UniformBaseStrategy(
        uncertain_param_type='If',
        uncertain_param_subgroup='ammonia',
        upper_scaling_factor = .5,
        lower_scaling_factor = .5,
        noise_interval={'min':.2, 'max':.2}
    ),
    processor.TriangluarBaseStrategy(
        uncertain_param_type='Cf',
        uncertain_param_subgroup=method,
        upper_scaling_factor = 0.15,
        lower_scaling_factor = 0.15,
        noise_interval={'min':.1, 'max':.1}
    ),
    processor.TriangluarBaseStrategy(
        uncertain_param_type='Var_bounds',
        uncertain_param_subgroup='upper_limit',
        upper_scaling_factor=.3,
        lower_scaling_factor=.3,
        noise_interval={'min':.2, 'max':.1}
    ),
    
    processor.ExpertKnowledgeStrategy(
        uncertain_param_type='If',
        uncertain_param_subgroup='ammonia',
        prob_metadata={
            (715, 23523): {'minimum':.1 ,'maximum':.8, 'uncertainty_type':4}, # CCS 200km pipeline 1000m deep | CO2 stored | RER --- Carbon dioxide, non-fossil | ('air',)
            (80, 23561): {'minimum':.0002 ,'maximum':.002, 'uncertainty_type':4}, # transport, supercritical carbon dioxide | transport, pipeline, supercritical carbon dioxide, 200 km without recompression | RER --- Carbon dioxide, fossil | ('air', 'urban air close to ground')
            # For the electricity we assume a triangular distribution with mode = the 75% decarbonized electricity (excel sheet) mix, min = 10% of mode, max = the current mix 
            # (81, 23537): {'loc':.0057, 'minimum':.00057 ,'maximum':.023, 'uncertainty_type':5}, # grid electricity | electricity, high voltage | RER --- Carbon dioxide, in air | ('natural resource', 'in air')
            # (82, 23537): {'loc':.0093, 'minimum':.00093 ,'maximum':.037, 'uncertainty_type':5}, # grid electricity | electricity, high voltage | RER --- Carbon dioxide, non-fossil | ('air', 'urban air close to ground')
            # (716, 23537): {'loc':.0663, 'minimum':.00663 ,'maximum':.265, 'uncertainty_type':5}, # grid electricity | electricity, high voltage | RER --- Carbon dioxide, fossil | ('air', 'non-urban air or from high stacks')
            # Playing around with electricity emissions, now assuming mode: ~90% decarbonized and min=Wind, and max=75% decarbonized
            (81, 23537): {'loc':.002, 'minimum':.00057 ,'maximum':.006, 'uncertainty_type':5}, # grid electricity | electricity, high voltage | RER --- Carbon dioxide, in air | ('natural resource', 'in air')
            (82, 23537): {'loc':.003, 'minimum':.00093 ,'maximum':.009, 'uncertainty_type':5}, # grid electricity | electricity, high voltage | RER --- Carbon dioxide, non-fossil | ('air', 'urban air close to ground')
            (716, 23537): {'loc':.023, 'minimum':.00663 ,'maximum':.069, 'uncertainty_type':5}, # grid electricity | electricity, high voltage | RER --- Carbon dioxide, fossil | ('air', 'non-urban air or from high stacks')
        
        }
    )
    ]

In [ ]:
pulpo_worker.import_and_filter_uncertainty_data(
    cutoff=0.,#0.000019,
    scaling_vector_strategy='constructed_demand',
    plot_results=True,
    plot_n_top_processes=19
)
pulpo_worker.apply_uncertainty_strategies(strategies=unc_strategies)
normal_metadata_env_cost, normal_metadata_var_bounds = pulpo_worker.create_CC_formulation(CC_env_cost=True, CC_var_bounds=['upper_limit'])

In [ ]:
lambda_epsilon_array = np.linspace(0.5,1, 20, endpoint=False)
results_CC = pulpo_worker.solve_CC_problem(lambda_epsilon_array, normal_metadata_env_cost, normal_metadata_var_bounds, plot_results=True)

In [ ]:
plots.plot_pareto_front(
    result_data_CC=results_CC, 
    cutoff_value=0.015, 
    method="\n".join(method.split("'")[1::2]), 
    process_map_metadata=pulpo_worker.lci_data['process_map_metadata'], 
    bbox_to_anchor=(0.45,-3.),
    cmap_name='tab20',
    group_act_by='process'
    )

## 6.3. Iterating the epistemic uncertainty

### Updating the epistemic uncertainty factors
- All uncertainty factors linked to the availability of the biomass 

### Modelling the updated uncertain factors
- Adding/Updating variable bounds to the biomass availabilities
- Adding triangular probability distribution on the var bounds 

# Supplementary code:


## Computing the environmental cost variance and mean of the alternative technologies

In [ ]:
import ast
import bw2calc
# Copied from "ParameterFilter.construct_scaling_vector_from_choices"
# Changed the self values and moved all computation to inner most loop and moved the demand dict initialization there
#--------------------
scaling_vector_collection = {}
for product, alternatives in pulpo_worker.choices.items():
    # demand_amount = self.result_data['Choices'][product]["Value"].sum()
    for alternative in alternatives:
        print(f'Conducting LCA for {alternative}')
        demand = {alternative: 1}
        # demand[alternative] = 1
        # Compute the scaling vector for the set constructed demand
        method_tuple = ast.literal_eval(next(iter(pulpo_worker.method)))
        lca = bw2calc.LCA(demand, method_tuple)
        lca = bw2calc.LCA(demand, method_tuple)
        lca.lci()
        # Map the scaling vector results of the LCI calculation back to the optimization results index structure
        index_mapper_df = pd.concat(
            [
                pd.DataFrame.from_dict(pulpo_worker.lci_data['process_map'], orient='index', columns=['opt_problem']),
                pd.DataFrame.from_dict(lca.product_dict, orient='index', columns=['lca'])
            ],
            axis=1
        ).set_index('opt_problem')
        reindex_supply_array_df = index_mapper_df.merge(pd.DataFrame(lca.supply_array,  columns=['supply_array']), how='left', left_on='lca', right_index=True)
        scaling_vector_series = reindex_supply_array_df['supply_array']
        scaling_vector_collection[alternative] = scaling_vector_series

Compute the impact results of each alternative from the environmental costs and with the environmental cost variance

In [ ]:
from scipy.stats import norm

In [ ]:
lambda_env_cost = 0.9
ppf_lambda_QB = norm.ppf(lambda_env_cost)
normal_env_cost_df = pd.DataFrame.from_dict(normal_metadata_env_cost, orient='index')[['loc', 'scale']]
normal_env_cost_df.index = normal_env_cost_df.index.droplevel(1)
impact_results = {}
for alternative, alternative_scaling_vector in scaling_vector_collection.items():
    alternative_scaling_vector = alternative_scaling_vector.reindex(normal_env_cost_df.index)
    impact_results[alternative] = {}
    impact_results[alternative]['mean'] = alternative_scaling_vector @ normal_env_cost_df['loc']
    impact_results[alternative]['std'] = alternative_scaling_vector @ normal_env_cost_df['scale']
    impact_results[alternative][f'lambda {lambda_env_cost}'] = alternative_scaling_vector @ (normal_env_cost_df['scale'] * ppf_lambda_QB)


In [ ]:
pd.DataFrame(impact_results).T